In [ ]:
import pandas as pd
# 讀取 Train70
train_set = pd.read_excel("PrediMOC_provide20250811_model1_grouped.xlsx", sheet_name='Train')

# 讀取 InternalTest30 工作表
internal_test_set = pd.read_excel("PrediMOC_provide20250811_model1_grouped.xlsx", sheet_name='InternalTest')

In [ ]:
train_set

In [ ]:
internal_test_set

In [ ]:
# 讀取 ExternalTest 工作表
External_set = pd.read_excel("PrediMOC_provide20250811_model1_grouped.xlsx", sheet_name='ExternalTest')

External_set

In [15]:
# 合併train_set internal_test_set External_set
train_set_copy = train_set.copy()
internal_test_set_copy = internal_test_set.copy()
External_set_copy = External_set.copy()


# 用 pd.concat()
all_data = pd.concat(
    [train_set_copy, internal_test_set_copy, External_set_copy],
    ignore_index=True
)

In [17]:
column_names_list = list(all_data.columns)
print(column_names_list)

['ID', 'Institute', 'Group', 'SMILoss4.2', 'Sex', 'Age', 'ECOG', 'Tumor_site', 'Stage', 'Chemotherapy', 'Smoking', 'pre_BMI', 'BMI_change', 'pre_SMI', 'Dermatitis', 'Mucositis', 'Xerostomia', 'Dysphagia', 'Pain', 'Anorexia', 'Nausea', 'Fatigue', 'Split']


不納入分析: ID Split

分組依據： Institute or Group

連續型: 'Age','pre_BMI', 'BMI_change','pre_SMI',

類別型: 'SMILoss4.2', 'Sex', 'ECOG', 'Tumor_site', 'Stage', 'Chemotherapy', 'Smoking', 'Dermatitis', 'Mucositis', 'Xerostomia', 'Dysphagia', 'Pain', 'Anorexia', 'Nausea', 'Fatigue'

# 描述性分析（依據Institute分）

In [32]:
import pandas as pd
import numpy as np
from scipy import stats

# 連續型變項
continuous_vars = ["Age","pre_BMI","BMI_change","pre_SMI",]

# 常態性判斷標準
ALPHA = 0.05
LARGE_SAMPLE_THRESHOLD = 50
SKEW_LIMIT = 2.0
KURT_LIMIT = 3.0

# 儲存最終建議
ttest_vars = []
mannwhitney_vars = []

print("Normality assessment for continuous variables")
print("=" * 55)

for col in continuous_vars:
    if col not in all_data.columns:
        print(f"{col}: column not found and was skipped.")
        continue

    # 依醫院來源分成兩個獨立群體
    group0 = all_data.loc[
        all_data["Institute"] == 0,
        col,
    ].dropna()

    group1 = all_data.loc[
        all_data["Institute"] == 1,
        col,
    ].dropna()

    group_normality = []

    for group_name, values in [
        ("MMH", group0),
        ("CCH", group1),
    ]:
        n = len(values)

        # 樣本數少於 3 時，無法可靠執行常態性評估
        if n < 3:
            group_normality.append(False)
            continue

        # 小樣本使用 Shapiro-Wilk test
        if n < LARGE_SAMPLE_THRESHOLD:
            _, p_value = stats.shapiro(values)
            is_normal = p_value > ALPHA

        # 大樣本依偏態與峰度判斷分布形狀
        else:
            skewness = stats.skew(
                values,
                bias=False,
            )

            kurtosis = stats.kurtosis(
                values,
                bias=False,
            )

            is_normal = (
                abs(skewness) < SKEW_LIMIT
                and abs(kurtosis) < KURT_LIMIT
            )

        group_normality.append(is_normal)

    # 兩組皆符合常態時建議 t-test；
    # 任一組不符合常態時建議 Mann-Whitney U test
    if all(group_normality):
        ttest_vars.append(col)
    else:
        mannwhitney_vars.append(col)

print(f"建議使用 t-test：{ttest_vars}")
print(f"建議使用 Mann-Whitney U test：{mannwhitney_vars}")

Normality assessment for continuous variables
建議使用 t-test：['Age', 'pre_BMI', 'BMI_change', 'pre_SMI']
建議使用 Mann-Whitney U test：[]


In [33]:
import numpy as np
import pandas as pd
from scipy import stats

# 類別型變項與分組欄位
categorical_vars = ["Muscle_loss","Sex","ECOG","Tumor_site","Stage","Chemotherapy"]

grouping_variable = "Institute"

# Cochran's rule：
# 期望次數小於 5 的儲存格不可超過全部儲存格的 20%
EXPECTED_COUNT_THRESHOLD = 5.0
PERCENTAGE_THRESHOLD = 0.20

print("Assessment of expected cell counts for categorical variables")
print("=" * 65)

for col in categorical_vars:
    if col not in all_data.columns:
        print(f"\n{col}: column not found and was skipped.")
        continue

    print(f"\nVariable: {col}")
    print("-" * 45)

    # 建立醫院來源與類別變項的列聯表
    contingency_table = pd.crosstab(
        all_data[grouping_variable],
        all_data[col],
    )

    print("Observed counts:")
    print(contingency_table)

    # 列聯表至少需要兩個群體及兩個類別
    if (
        contingency_table.shape[0] < 2
        or contingency_table.shape[1] < 2
    ):
        print("Recommendation: insufficient table dimensions.")
        continue

    try:
        # 取得卡方檢定所使用的期望次數
        _, _, _, expected_freq = stats.chi2_contingency(
            contingency_table
        )

        expected_table = pd.DataFrame(
            expected_freq,
            index=contingency_table.index,
            columns=contingency_table.columns,
        ).round(2)

        print("\nExpected counts:")
        print(expected_table)

        # 計算期望次數小於 5 的儲存格比例
        total_cells = expected_freq.size
        cells_below_threshold = np.sum(
            expected_freq < EXPECTED_COUNT_THRESHOLD
        )
        percentage_below = (
            cells_below_threshold / total_cells
        )

        print(
            f"\nCells with expected count < 5: "
            f"{cells_below_threshold}/{total_cells} "
            f"({percentage_below:.1%})"
        )

        # 依 Cochran's 20% rule 選擇統計檢定
        if percentage_below > PERCENTAGE_THRESHOLD:
            print("Recommendation: Fisher's exact test.")
        else:
            print("Recommendation: Chi-square test.")

            # 即使未超過 20%，任何期望次數小於 1
            # 仍表示卡方近似可能不可靠
            if np.any(expected_freq < 1):
                print(
                    "Warning: at least one expected count is below 1; "
                    "consider Fisher's exact test."
                )

    except ValueError as error:
        print(f"Chi-square calculation failed: {error}")
        print("Recommendation: Fisher's exact test.")

Assessment of expected cell counts for categorical variables

Muscle_loss: column not found and was skipped.

Variable: Sex
---------------------------------------------
Observed counts:
Sex         1    2
Institute         
0          35  296
1          78  494

Expected counts:
Sex            1       2
Institute               
0          41.42  289.58
1          71.58  500.42

Cells with expected count < 5: 0/4 (0.0%)
Recommendation: Chi-square test.

Variable: ECOG
---------------------------------------------
Observed counts:
ECOG         0    1
Institute          
0          238   93
1          388  184

Expected counts:
ECOG            0       1
Institute                
0          229.46  101.54
1          396.54  175.46

Cells with expected count < 5: 0/4 (0.0%)
Recommendation: Chi-square test.

Variable: Tumor_site
---------------------------------------------
Observed counts:
Tumor_site    1    2    3
Institute                
0           105  126  100
1           222  224  1

In [34]:
import pandas as pd
from tableone import TableOne


# 2.1 定義變數列表
continuous_vars = [
    'Age', 'pre_BMI', 'BMI_change', 'pre_SMI'
]
categorical_vars = [
    'SMILoss4.2', 'Sex', 'ECOG', 'Tumor_site', 'Stage', 
    'Chemotherapy', 'Smoking', 'Dermatitis', 'Mucositis', 
    'Xerostomia', 'Dysphagia', 'Pain', 'Anorexia', 
    'Nausea', 'Fatigue'
]

# 依醫院來源進行分組
grouping_variable = "Institute"

# 將數值編碼轉換為可讀的醫院名稱
all_data["Institute_Label"] = all_data[grouping_variable].map({
    0: "MMH",
    1: "CCH",
})

# Table 1 納入的全部變項
all_table_vars = continuous_vars + categorical_vars

# 根據先前的分布評估，所有連續變項均以常態資料呈現：
# Mean (SD)，並採用 t-test 進行兩組比較
nonnormal_vars = []

# 檢查兩個醫院的樣本數
print("Sample size by institute")
print("=" * 35)
print(all_data["Institute_Label"].value_counts())
print()

# 建立描述性統計表
table1 = TableOne(
    data=all_data,
    columns=all_table_vars,
    categorical=categorical_vars,
    groupby="Institute_Label",
    nonnormal=nonnormal_vars,
    pval=True,
    missing=True,
    overall=True,
    decimals=2,
    htest_name=True,
)

# 顯示 Table 1
print(table1.tabulate(tablefmt="fancy_grid"))

Sample size by institute
Institute_Label
CCH    572
MMH    331
Name: count, dtype: int64

╒═══════════════════════╤════╤═══════════╤═══════════════╤═══════════════╤═══════════════╤═══════════╤════════════════╕
│                       │    │ Missing   │ Overall       │ CCH           │ MMH           │ P-Value   │ Test           │
╞═══════════════════════╪════╪═══════════╪═══════════════╪═══════════════╪═══════════════╪═══════════╪════════════════╡
│ n                     │    │           │ 903           │ 572           │ 331           │           │                │
├───────────────────────┼────┼───────────┼───────────────┼───────────────┼───────────────┼───────────┼────────────────┤
│ Age, mean (SD)        │    │ 0         │ 54.74 (10.37) │ 54.54 (10.19) │ 55.08 (10.68) │ 0.455     │ Welch’s T-test │
├───────────────────────┼────┼───────────┼───────────────┼───────────────┼───────────────┼───────────┼────────────────┤
│ pre_BMI, mean (SD)    │    │ 0         │ 23.36 (3.48)  │ 23.22 (3.21

In [16]:
table1.to_excel("descriptive_statistics_old_data_byInstitute.xlsx")
print("\n表格已匯出至 'descriptive_statistics_old_data_byInstitute.xlsx'")


表格已匯出至 'descriptive_statistics_old_data_byInstitute.xlsx'


# 描述性分析（依據Group分）

In [35]:
import pandas as pd
import numpy as np
from scipy import stats

# 連續型變項
continuous_vars = ['Age','pre_BMI', 'BMI_change','pre_SMI']

# 常態性判斷標準
ALPHA = 0.05
LARGE_SAMPLE_THRESHOLD = 50
SKEW_LIMIT = 2.0
KURT_LIMIT = 7.0

# Group 編碼與資料集名稱
group_labels = {
    0: "Training",
    1: "Internal Validation",
    2: "External Validation",
}

# 儲存最終檢定建議
anova_vars = []
kruskal_vars = []

for col in continuous_vars:
    if col not in all_data.columns:
        print(f"{col}: column not found and was skipped.")
        continue

    group_normality = []

    for group_idx, group_name in group_labels.items():
        # 取得該資料集的有效觀測值
        group_data = all_data.loc[
            all_data["Group"] == group_idx,
            col,
        ].dropna()

        n = len(group_data)

        # 樣本數不足時，保守視為不符合常態
        if n < 3:
            group_normality.append(False)
            continue

        # 小樣本使用 Shapiro-Wilk test
        if n < LARGE_SAMPLE_THRESHOLD:
            _, p_value = stats.shapiro(group_data)
            is_normal = p_value > ALPHA

        # 大樣本依偏態及峰度評估分布形狀
        else:
            skewness = stats.skew(
                group_data,
                bias=False,
            )

            kurtosis = stats.kurtosis(
                group_data,
                bias=False,
            )

            is_normal = (
                abs(skewness) < SKEW_LIMIT
                and abs(kurtosis) < KURT_LIMIT
            )

        group_normality.append(is_normal)

    # 三個資料集均符合條件時使用 ANOVA；
    # 任一資料集不符合條件時使用 Kruskal-Wallis test
    if all(group_normality):
        anova_vars.append(col)
    else:
        kruskal_vars.append(col)

print(f"建議使用 ANOVA（Mean ± SD）：{anova_vars}")
print(
    "建議使用 Kruskal-Wallis test（Median [IQR]）："
    f"{kruskal_vars}"
)

建議使用 ANOVA（Mean ± SD）：['Age', 'pre_BMI', 'BMI_change', 'pre_SMI']
建議使用 Kruskal-Wallis test（Median [IQR]）：[]


In [36]:
import pandas as pd
import numpy as np
from scipy import stats

# 模型一的類別型變項
categorical_vars = ['SMILoss4.2', 'Sex', 'ECOG', 'Tumor_site', 'Stage', 'Chemotherapy', 'Smoking', 'Dermatitis', 'Mucositis', 
                    'Xerostomia', 'Dysphagia', 'Pain', 'Anorexia', 'Nausea', 'Fatigue']

# 建立資料集分組標籤
all_data["Group_Label"] = all_data["Group"].map({
    0: "Training",
    1: "Internal validation",
    2: "External validation",
})

# 儲存最終檢定建議
chi_square_vars = []
fisher_vars = []
skipped_vars = []

for var in categorical_vars:
    if var not in all_data.columns:
        skipped_vars.append(var)
        continue

    # 建立類別變項與資料集來源的列聯表
    contingency_table = pd.crosstab(
        all_data[var],
        all_data["Group_Label"],
    )

    rows, columns = contingency_table.shape

    # 列聯表至少需要兩個類別及兩個資料集
    if rows < 2 or columns < 2:
        skipped_vars.append(var)
        continue

    try:
        # 計算卡方檢定所使用的期望次數
        _, _, _, expected = stats.chi2_contingency(
            contingency_table
        )

        total_cells = expected.size
        cells_below_5 = np.sum(expected < 5)
        percentage_below_5 = (
            cells_below_5 / total_cells
        ) * 100

        # Cochran's rule：
        # 超過 20% 的儲存格期望次數小於 5，
        # 或任何儲存格期望次數小於 1，
        violates_cochran_rule = (
            percentage_below_5 > 20
            or np.min(expected) < 1
        )

        if violates_cochran_rule:
            fisher_vars.append(var)
        else:
            chi_square_vars.append(var)

    except ValueError:
        fisher_vars.append(var)

print(f"建議使用 Chi-square test：{chi_square_vars}")
print(
    "建議使用 Fisher's exact test 或合併類別："
    f"{fisher_vars}"
)

if skipped_vars:
    print(f"未執行檢定：{skipped_vars}")

建議使用 Chi-square test：['SMILoss4.2', 'Sex', 'ECOG', 'Tumor_site', 'Stage', 'Chemotherapy', 'Smoking', 'Dermatitis', 'Mucositis', 'Xerostomia', 'Dysphagia', 'Pain', 'Anorexia', 'Nausea', 'Fatigue']
建議使用 Fisher's exact test 或合併類別：[]


In [37]:
import pandas as pd
import numpy as np
from tableone import TableOne

# 模型一的連續型變項
continuous_vars = ['Age','pre_BMI', 'BMI_change','pre_SMI']
categorical_vars = ['SMILoss4.2', 'Sex', 'ECOG', 'Tumor_site', 'Stage', 'Chemotherapy', 'Smoking', 'Dermatitis', 'Mucositis', 
                    'Xerostomia', 'Dysphagia', 'Pain', 'Anorexia', 'Nausea', 'Fatigue']

# 建立資料集分組標籤
all_data["Group_Label"] = all_data["Group"].map({
    0: "Training",
    1: "Internal validation",
    2: "External validation",
})

# Table 1 納入的全部變項
all_table_vars = continuous_vars + categorical_vars

# 確保連續型欄位為 numeric；
# 無法轉換的內容會設為缺失值 NaN
for col in continuous_vars:
    if col in all_data.columns:
        all_data[col] = pd.to_numeric(
            all_data[col],
            errors="coerce",
        )

# 根據分布評估結果，所有連續變項均以常態資料呈現：
# Mean (SD)，並使用 One-way ANOVA 比較三個資料集
nonnormal_vars = []

# 檢查各資料集的樣本數
print("Sample size by dataset")
print("=" * 40)
print(all_data["Group_Label"].value_counts())
print()

# 建立描述性統計表
table1_dataset_comparison = TableOne(
    data=all_data,
    columns=all_table_vars,
    categorical=categorical_vars,
    groupby="Group_Label",
    nonnormal=nonnormal_vars,
    pval=True,
    missing=True,
    overall=True,
    decimals=2,
    htest_name=True,
)

table1_dataset_comparison

Sample size by dataset
Group_Label
Training               400
External validation    331
Internal validation    172
Name: count, dtype: int64



Grouped by Group_Label                                                                                             
                                       Missing        Overall External validation Internal validation       Training P-Value           Test
n                                                         903                 331                 172            400                       
Age, mean (SD)                               0  54.74 (10.37)       55.08 (10.68)        54.32 (9.36)  54.63 (10.54)   0.712  One-way ANOVA
pre_BMI, mean (SD)                           0   23.36 (3.48)        23.60 (3.90)        23.13 (3.28)   23.25 (3.18)   0.261  One-way ANOVA
BMI_change, mean (SD)                        0   -2.42 (3.98)        -1.85 (5.24)        -2.73 (2.95)   -2.77 (3.00)   0.004  One-way ANOVA
pre_SMI, mean (SD)                           0   51.68 (8.30)        52.27 (8.66)        50.93 (7.65)   51.51 (8.26)   0.197  One-way ANOVA
SMILoss4.2, n (%)     0                           678 (75.08)         250 (75.53)         129 (75.00)    299 (74.75)   0.971    Chi-squared
                      1                           225 (24.92)          81 (24.47)          43 (25.00)    101 (25.25)                       
Sex, n (%)            1                           113 (12.51)          35 (10.57)          21 (12.21)     57 (14.25)   0.324    Chi-squared
                      2                           790 (87.49)         296 (89.43)         151 (87.79)    343 (85.75)                       
ECOG, n (%)           0                           626 (69.32)         238 (71.90)         126 (73.26)    262 (65.50)   0.081    Chi-squared
                      1                           277 (30.68)          93 (28.10)          46 (26.74)    138 (34.50)                       
Tumor_site, n (%)     1                           327 (36.21)         105 (31.72)          74 (43.02)    148 (37.00)   0.027    Chi-squared
                      2                           350 (38.76)         126 (38.07)          66 (38.37)    158 (39.50)                       
                      3                           226 (25.03)         100 (30.21)          32 (18.60)     94 (23.50)                       
Stage, n (%)          0                           151 (16.72)          51 (15.41)          29 (16.86)     71 (17.75)   0.699    Chi-squared
                      1                           752 (83.28)         280 (84.59)         143 (83.14)    329 (82.25)                       
Chemotherapy, n (%)   0                           349 (38.65)         130 (39.27)          69 (40.12)    150 (37.50)   0.805    Chi-squared
                      1                           554 (61.35)         201 (60.73)         103 (59.88)    250 (62.50)                       
Smoking, n (%)        0                           300 (33.22)         114 (34.44)          53 (30.81)    133 (33.25)   0.715    Chi-squared
                      1                           603 (66.78)         217 (65.56)         119 (69.19)    267 (66.75)                       
Dermatitis, n (%)     1                           461 (51.05)         175 (52.87)          88 (51.16)    198 (49.50)   0.354    Chi-squared
                      2                           341 (37.76)         120 (36.25)          71 (41.28)    150 (37.50)                       
                      3                           101 (11.18)          36 (10.88)           13 (7.56)     52 (13.00)                       
Mucositis, n (%)      1                           219 (24.25)          81 (24.47)          41 (23.84)     97 (24.25)   0.951    Chi-squared
                      2                           493 (54.60)         180 (54.38)          98 (56.98)    215 (53.75)                       
                      3                           191 (21.15)          70 (21.15)          33 (19.19)     88 (22.00)                       
Xerostomia, n (%)     1                           204 (22.59)          77 (23.26)          33 (19.19)   

In [12]:
table1_dataset_comparison.to_excel("descriptive_statistics_old_data_byGroup.xlsx")
print("\n表格已匯出至 'descriptive_statistics_old_data_byGroup.xlsx'")


表格已匯出至 'descriptive_statistics_old_data_byGroup.xlsx'


# 針對 有無肌肉流失個案作比較

In [38]:
import pandas as pd
from scipy import stats

def compare_muscle_loss_groups(
    df,
    label_col,
    continuous_vars,
    categorical_vars,
    dataset_name="Dataset",
):
    print(f"\n{'=' * 18} {dataset_name} {'=' * 18}")

    results = []

    # 確認結果欄位包含兩個群體
    groups = sorted(df[label_col].dropna().unique())

    if len(groups) != 2:
        raise ValueError(
            f"{label_col} 必須包含兩組，目前為：{groups}"
        )

    group0, group1 = groups

    # 連續型變項：
    # 以 Median [IQR] 呈現，使用 Mann-Whitney U test
    for col in continuous_vars:
        if col not in df.columns:
            continue

        values0 = df.loc[
            df[label_col] == group0,
            col,
        ].dropna()

        values1 = df.loc[
            df[label_col] == group1,
            col,
        ].dropna()

        _, p_value = stats.mannwhitneyu(
            values0,
            values1,
            alternative="two-sided",
        )

        results.append({
            "Variable": col,
            "Variable type": "Continuous",
            "Summary": "Median [IQR]",
            "Test": "Mann-Whitney U test",
            "P-value": p_value,
        })

    # 類別型變項：
    # 以 n (%) 呈現，使用 Chi-square test
    for col in categorical_vars:
        if col not in df.columns or col == label_col:
            continue

        contingency_table = pd.crosstab(
            df[col],
            df[label_col],
        )

        if (
            contingency_table.shape[0] < 2
            or contingency_table.shape[1] < 2
        ):
            continue

        _, p_value, _, _ = stats.chi2_contingency(
            contingency_table
        )

        results.append({
            "Variable": col,
            "Variable type": "Categorical",
            "Summary": "n (%)",
            "Test": "Chi-square test",
            "P-value": p_value,
        })

    results_df = pd.DataFrame(results)

    # 加入統計顯著性標記
    results_df["Significance"] = results_df["P-value"].apply(
        lambda p: (
            "***" if p < 0.001
            else "**" if p < 0.01
            else "*" if p < 0.05
            else ""
        )
    )

    display_df = results_df.copy()
    display_df["P-value"] = display_df["P-value"].map(
        lambda p: "<0.001" if p < 0.001 else f"{p:.4f}"
    )

    display(display_df)
    return results_df

In [39]:
train_comparison = compare_muscle_loss_groups(
    train_set,
    LABEL_COL,
    continuous_vars,
    categorical_vars,
    dataset_name="Training Set",
)

internal_comparison = compare_muscle_loss_groups(
    internal_test_set,
    LABEL_COL,
    continuous_vars,
    categorical_vars,
    dataset_name="Internal Validation Set",
)

external_comparison = compare_muscle_loss_groups(
    External_set,
    LABEL_COL,
    continuous_vars,
    categorical_vars,
    dataset_name="External Validation Set",
)


================== Training Set ==================


,Variable,Variable type,Summary,Test,P-value,Significance
0,Age,Continuous,Median [IQR],Mann-Whitney U test,0.4124,
1,pre_BMI,Continuous,Median [IQR],Mann-Whitney U test,0.0649,
2,BMI_change,Continuous,Median [IQR],Mann-Whitney U test,0.7037,
3,pre_SMI,Continuous,Median [IQR],Mann-Whitney U test,<0.001,***
4,Sex,Categorical,n (%),Chi-square test,0.9718,
5,ECOG,Categorical,n (%),Chi-square test,0.0099,**
6,Tumor_site,Categorical,n (%),Chi-square test,0.0350,*
7,Stage,Categorical,n (%),Chi-square test,0.0017,**
8,Chemotherapy,Categorical,n (%),Chi-square test,<0.001,***
9,Smoking,Categorical,n (%),Chi-square test,0.3186,



================== Internal Validation Set ==================


,Variable,Variable type,Summary,Test,P-value,Significance
0,Age,Continuous,Median [IQR],Mann-Whitney U test,0.2836,
1,pre_BMI,Continuous,Median [IQR],Mann-Whitney U test,0.3178,
2,BMI_change,Continuous,Median [IQR],Mann-Whitney U test,0.5860,
3,pre_SMI,Continuous,Median [IQR],Mann-Whitney U test,0.0724,
4,Sex,Categorical,n (%),Chi-square test,0.6867,
5,ECOG,Categorical,n (%),Chi-square test,0.0054,**
6,Tumor_site,Categorical,n (%),Chi-square test,0.8420,
7,Stage,Categorical,n (%),Chi-square test,0.0068,**
8,Chemotherapy,Categorical,n (%),Chi-square test,<0.001,***
9,Smoking,Categorical,n (%),Chi-square test,0.0031,**



================== External Validation Set ==================


,Variable,Variable type,Summary,Test,P-value,Significance
0,Age,Continuous,Median [IQR],Mann-Whitney U test,0.0016,**
1,pre_BMI,Continuous,Median [IQR],Mann-Whitney U test,0.8995,
2,BMI_change,Continuous,Median [IQR],Mann-Whitney U test,<0.001,***
3,pre_SMI,Continuous,Median [IQR],Mann-Whitney U test,0.1866,
4,Sex,Categorical,n (%),Chi-square test,0.6975,
5,ECOG,Categorical,n (%),Chi-square test,<0.001,***
6,Tumor_site,Categorical,n (%),Chi-square test,0.3728,
7,Stage,Categorical,n (%),Chi-square test,0.0342,*
8,Chemotherapy,Categorical,n (%),Chi-square test,0.0031,**
9,Smoking,Categorical,n (%),Chi-square test,0.0052,**


In [40]:
import pandas as pd
from scipy import stats

# 結果欄位與不納入分析的識別欄位
LABEL = "SMILoss4.2"

EXCLUDE_COLS = [
    "ID",
    "Institute",
    "Group",
    "Split",
    LABEL,
]

# 依訓練集自動區分連續型與類別型變項
CAT_FEATURES = []
CONT_FEATURES = []

for col in train_set.columns:
    if col in EXCLUDE_COLS:
        continue

    # 數值型且唯一值超過 10 時視為連續型，
    # 其餘變項視為類別型
    if (
        pd.api.types.is_numeric_dtype(train_set[col])
        and train_set[col].nunique(dropna=True) > 10
    ):
        CONT_FEATURES.append(col)
    else:
        CAT_FEATURES.append(col)

print("Feature classification")
print(f"Categorical variables: {CAT_FEATURES}")
print(f"Continuous variables: {CONT_FEATURES}")

Feature classification
Categorical variables: ['Sex', 'ECOG', 'Tumor_site', 'Stage', 'Chemotherapy', 'Smoking', 'Dermatitis', 'Mucositis', 'Xerostomia', 'Dysphagia', 'Pain', 'Anorexia', 'Nausea', 'Fatigue']
Continuous variables: ['Age', 'pre_BMI', 'BMI_change', 'pre_SMI']


In [41]:
def get_table_content(
    df,
    label_col,
    categorical_cols,
    continuous_cols,
):
    table_rows = []

    # 只保留具有結果標籤的個案，並確認 0/1 分組
    valid_df = df.dropna(subset=[label_col]).copy()
    groups = sorted(valid_df[label_col].unique())

    if len(groups) != 2:
        raise ValueError(
            f"{label_col} 必須包含兩組，目前為：{groups}"
        )

    n_info = {
        group: (valid_df[label_col] == group).sum()
        for group in groups
    }

    # 連續型變項：
    # Median (Q1-Q3) 與 Mann-Whitney U test
    for col in continuous_cols:
        row = {
            "Characteristics": col,
        }

        group_values = []

        for group in groups:
            values = valid_df.loc[
                valid_df[label_col] == group,
                col,
            ].dropna()

            group_values.append(values)

            if values.empty:
                row[f"G{group}_val"] = "N/A"
            else:
                q1 = values.quantile(0.25)
                median = values.median()
                q3 = values.quantile(0.75)

                row[f"G{group}_val"] = (
                    f"{median:.2f} "
                    f"({q1:.2f}-{q3:.2f})"
                )

        _, p_value = stats.mannwhitneyu(
            group_values[0],
            group_values[1],
            alternative="two-sided",
        )

        row["p"] = (
            f"{p_value:.3f}"
            if p_value >= 0.001
            else "<0.001"
        )

        table_rows.append(row)

    # 類別型變項：
    # n (%) 與 Chi-square test
    for col in categorical_cols:
        contingency_table = pd.crosstab(
            valid_df[col],
            valid_df[label_col],
        )

        _, p_value, _, _ = stats.chi2_contingency(
            contingency_table
        )

        p_string = (
            f"{p_value:.3f}"
            if p_value >= 0.001
            else "<0.001"
        )

        # 變項名稱列顯示整體 p 值
        table_rows.append({
            "Characteristics": col,
            "p": p_string,
        })

        # 各類別列顯示人數與組內百分比
        categories = sorted(
            valid_df[col].dropna().unique()
        )

        for category in categories:
            category_row = {
                "Characteristics": f"  {category}",
                "p": "",
            }

            for group in groups:
                count = (
                    (valid_df[label_col] == group)
                    & (valid_df[col] == category)
                ).sum()

                percentage = (
                    count / n_info[group]
                ) * 100

                category_row[f"G{group}_val"] = (
                    f"{count} ({percentage:.1f}%)"
                )

            table_rows.append(category_row)

    return pd.DataFrame(table_rows), n_info

In [42]:
res_train, n_train = get_table_content(
    train_set,
    LABEL,
    CAT_FEATURES,
    CONT_FEATURES,
)

res_internal, n_internal = get_table_content(
    internal_test_set,
    LABEL,
    CAT_FEATURES,
    CONT_FEATURES,
)

res_external, n_external = get_table_content(
    External_set,
    LABEL,
    CAT_FEATURES,
    CONT_FEATURES,
)

In [43]:
final_table = pd.concat(
    [
        res_train[
            ["Characteristics", "G0_val", "G1_val", "p"]
        ],
        res_internal[
            ["G0_val", "G1_val", "p"]
        ],
        res_external[
            ["G0_val", "G1_val", "p"]
        ],
    ],
    axis=1,
)

final_table.columns = [
    "Characteristics",
    f"Loss (-), Training (n={n_train[0]})",
    f"Loss (+), Training (n={n_train[1]})",
    "p, Training",
    f"Loss (-), Internal (n={n_internal[0]})",
    f"Loss (+), Internal (n={n_internal[1]})",
    "p, Internal",
    f"Loss (-), External (n={n_external[0]})",
    f"Loss (+), External (n={n_external[1]})",
    "p, External",
]

display(final_table)

,Characteristics,"Loss (-), Training (n=299)","Loss (+), Training (n=101)","p, Training","Loss (-), Internal (n=129)","Loss (+), Internal (n=43)","p, Internal","Loss (-), External (n=250)","Loss (+), External (n=81)","p, External"
0,Age,54.00 (46.00-62.00),55.00 (49.00-60.00),0.412,55.00 (47.00-60.00),56.00 (50.00-62.00),0.284,54.00 (46.25-61.00),58.00 (52.00-64.00),0.002
1,pre_BMI,23.21 (21.15-25.45),22.62 (20.80-24.31),0.065,22.70 (21.39-25.08),22.23 (20.07-25.29),0.318,23.14 (20.97-25.77),23.23 (21.08-26.22),0.900
2,BMI_change,-3.31 (-5.06--0.78),-3.44 (-4.62--0.67),0.704,-3.46 (-4.61--0.27),-3.88 (-4.55--1.75),0.586,-1.57 (-4.00-2.46),-4.29 (-8.33--1.54),<0.001
3,pre_SMI,51.15 (46.47-58.07),47.74 (44.27-52.69),<0.001,50.42 (47.01-55.85),46.03 (42.36-54.50),0.072,51.67 (46.44-57.22),51.15 (45.23-56.04),0.187
4,Sex,NaN,NaN,0.972,NaN,NaN,0.687,NaN,NaN,0.697
5,1,42 (14.0%),15 (14.9%),,17 (13.2%),4 (9.3%),,25 (10.0%),10 (12.3%),
6,2,257 (86.0%),86 (85.1%),,112 (86.8%),39 (90.7%),,225 (90.0%),71 (87.7%),
7,ECOG,NaN,NaN,0.010,NaN,NaN,0.005,NaN,NaN,<0.001
8,0,207 (69.2%),55 (54.5%),,102 (79.1%),24 (55.8%),,197 (78.8%),41 (50.6%),
9,1,92 (30.8%),46 (45.5%),,27 (20.9%),19 (44.2%),,53 (21.2%),40 (49.4%),


In [16]:
# 匯出 Excel
final_table.to_excel("正負樣本Statistical_Analysis.xlsx", index=False)
print("表格已產出至 Table2_Statistical_Analysis.xlsx")

表格已產出至 Table2_Statistical_Analysis.xlsx
